# OmniFall 2 — Videos

The annotations are redistributable; most of the videos are not. They belong to
the original authors, so this package fetches them from the original sources
rather than mirroring them.

```bash
pip install 'omnifall[video]'
```

In [ ]:
import omnifall

for name, prepared in omnifall.status().items():
    print(f"{name:11s} {'present' if prepared else 'missing'}")

## Where the videos come from

Three of the ten components can be fetched fully automatically. Three more can be
downloaded but still need a conversion step. Four have to be obtained by hand,
either because the site blocks automation or because access is by request.

In [ ]:
for name, src in omnifall.SOURCES.items():
    size = f"{src.approx_bytes / 2**30:.1f} GiB" if src.approx_bytes else "-"
    how = "automatic" if src.automatable else "manual"
    print(f"{name:11s} {src.kind:8s} {size:>9s}  {how}")

`omnifall sources <name>` prints the download route, the licence and the
citation for any component, and the manual steps where there is no automated
route.

## Option A — you already have the videos

Point `OMNIFALL_ROOT` at a directory laid out as
`{root}/{dataset}/video/{path}.mp4` and nothing is downloaded at all:

```bash
export OMNIFALL_ROOT=/path/to/data
```

Individual components can be overridden with
`OMNIFALL_VIDEO_ROOT__<dataset>`, which takes precedence.

## Option B — let the package fetch them

**OF-Syn** is hosted on the Hub as a single archive (9.7 GB) and needs no
agreement:

```python
omnifall.prepare("of-syn")
```

**OF-ItW** uses videos from the OOPS dataset, which is distributed under
CC BY-NC-SA 4.0 for non-commercial research. `prepare` shows the licence notice
and asks for confirmation; pass `consent=True` to accept non-interactively. It
streams the ~45 GB archive and keeps only the 818 videos OmniFall uses (~2.6 GB):

```python
omnifall.prepare("OOPS")
```

Both are idempotent — already-prepared components return immediately. The cells
below are deliberately not executed here, because between them they move about
55 GB.

In [ ]:
# Uncomment to run. Both are safe to re-run and resume.
# omnifall.prepare("of-syn")
# omnifall.prepare("OOPS", consent=True)

## Attaching video paths

`video=True` adds a `video` column of absolute paths.

In [ ]:
ds = omnifall.load("le2i-cs", split="validation", video=True)
ds[0]["video"]

A missing file becomes `None` and you get one aggregated warning, never one per
row. In a training script you usually want the opposite — fail immediately:

```python
ds = omnifall.load("cs", video=True, strict=True)   # raises MissingVideosError
```

To see what is resolvable without loading anything:

In [ ]:
report = omnifall.resolution_report(omnifall.load("le2i-cs", split="validation"))
print(report.summary())

## Checking a local copy

`verify` compares a directory against the published label file. It reports
missing files, zero-byte files, and files the labels never reference.

In [ ]:
r = omnifall.verify("le2i")
print(f"required={r.required} present={r.present} "
      f"missing={len(r.missing)} unreferenced={r.extra}")

Unreferenced files are normal for some components. `cmdfall` ships the same
recordings twice — 384 continuous videos, which OmniFall annotates, and 1,052
pre-cut clips, which it does not.

## Decoding a segment

A row is a segment, so decoding means seeking to `start` and taking frames from
inside `[start, end]` — not decoding the whole file.

In [ ]:
from omnifall._decode import decode_segment, probe

row = ds[0]
meta = probe(row["video"])
print(f"{meta.width}x{meta.height} @ {meta.fps:.1f} fps, {meta.duration:.2f} s")
print(f"segment: {row['start']:.2f}–{row['end']:.2f} s "
      f"= {omnifall.IDX2LABEL[row['label']]}")

frames = decode_segment(
    row["video"],
    start=row["start"], end=row["end"],
    num_frames=8, target_fps=15.0,
    sampling="uniform",
)
frames.shape, frames.dtype

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
for ax, f in zip(axes, frames):
    ax.imshow(f)
    ax.axis("off")
fig.suptitle(omnifall.IDX2LABEL[row["label"]])
plt.tight_layout()

`sampling` controls how the frames are chosen inside the segment:

| Value | Behaviour | Use for |
|---|---|---|
| `"uniform"` | spread evenly over the whole segment, deterministic | evaluation |
| `"center"` | a window centred in the segment, deterministic | evaluation |
| `"random"` | a window at a random offset | training |

A handful of published segments in the multi-view components are annotated past
the end of their video. `decode_segment` clamps to the available footage and pads
by repeating the last frame, so you always get exactly `num_frames`. It never
raises for that — see KNOWN_PITFALLS.md on the dataset page.

## Next

- **[03_training.ipynb](03_training.ipynb)** — DataLoaders and 🤗 `transformers`